# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² protein abundance dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and published via this URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

md = dataset.metadata
print(f"{md.name}\n\n{md.description}\n")
print(f"Published: {md.datePublished}\nVersion: {md.version}\nLicense: {md.license}")

## 2. Data Overview
Let's inspect what record sets and field definitions exist in the Croissant package. This helps to understand the structure of the dataset and discover IDs to use in following steps.

In [ ]:
# List available record sets, fields, and their @ids
print('Available record sets:')
for recset in dataset.record_sets:
    print(f"- @id: {recset.id}, name: {recset.name}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all available record set @ids for use below
record_sets = [recset.id for recset in dataset.record_sets]
print(f'Record set @ids: {record_sets}\n')

dataframes = {}
for recset_id in record_sets:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Loaded {df.shape[0]} records for record set: {recset_id}")
    print(f"  Example columns: {df.columns[:8].tolist()}")

Next, let's look at an example record set DataFrame. (The actual `@id` string may differ in future versions; use one shown above, e.g. 'cr:RecordSet/proteinTable')

In [ ]:
# Pick a record set to preview -- update with actual @id as needed
chosen_record_set = record_sets[0]
print(f'Selected record set for preview: {chosen_record_set}')
print('Available fields:')
print(dataframes[chosen_record_set].columns.tolist())
dataframes[chosen_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply some basic analysis to numeric fields, such as filtering, normalizing, and grouping.

In [ ]:
# Choose a numeric field in the selected record set
# You may need to adjust the field name based on .columns output above. Example: 'coverage_percent', 'peptide_count', etc.
numeric_field = None
for col in dataframes[chosen_record_set].columns:
    # Heuristics for likely numeric fields
    if col.lower().startswith('coverage') or col.lower().startswith('mw') or col.lower().endswith('count') or col.lower().startswith('abundance'):
        numeric_field = col
        break

if numeric_field is None:
    print('No obvious numeric field found. Please adjust selection.')
else:
    threshold = dataframes[chosen_record_set][numeric_field].dropna().mean()
    filtered_df = dataframes[chosen_record_set][dataframes[chosen_record_set][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value): {len(filtered_df)} records")

    # Normalize numeric_field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a common field
    group_by_field = None
    for col in dataframes[chosen_record_set].columns:
        if col != numeric_field and ('accession' in col.lower() or 'description' in col.lower() or 'gene' in col.lower()):
            group_by_field = col
            break

    if group_by_field:
        grouped_df = filtered_df.groupby(group_by_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"\nGrouped by '{group_by_field}', mean {numeric_field} per group:")
        print(grouped_df.head())
    else:
        print('No obvious grouping field found.')

## 5. Visualization
Visualize the distribution of a numeric value or the relationship between fields in the chosen record set using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[chosen_record_set][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} in {chosen_record_set}')
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_by_field:
        top_groups = filtered_df[group_by_field].value_counts().nlargest(5).index
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_by_field, y=numeric_field, data=filtered_df[filtered_df[group_by_field].isin(top_groups)])
        plt.title(f'{numeric_field} by Top {group_by_field}')
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a FAIR² mass spectrometry dataset via Croissant metadata, review its structure, extract dataframes using record set and field `@id`s, and perform basic exploratory data analysis (EDA) steps using Python data science libraries.

Key findings and possible next steps:
- The dataset provides rich protein-level data suitable for abundance and modification analysis.
- Heuristic field selection enables initial EDA even in complex, generic Croissant packages.
- For rigorous downstream analysis, consult the field documentation by inspecting schema field `@id`s and metadata further.

For more advanced analysis, consider cross-referencing protein identifiers with UniProt, analyzing by sample or experimental condition fields, and performing statistical tests on normalized/filtered features.